## 1. Modules

In [236]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.ndimage import distance_transform_edt

## 2. Mounting drive

In [237]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Dataset

In [238]:
DRIVE_EXT = "/content/drive/My Drive/"
FOLDER_EXT = f"{DRIVE_EXT}/RVG_Merged_Dataset"
IMAGE_DIR = f"{FOLDER_EXT}/images"
MASK_DIR = f"{FOLDER_EXT}/masks_cleaned"

print("All extensions...")
print(DRIVE_EXT)
print(FOLDER_EXT)
print(IMAGE_DIR)
print(MASK_DIR)

All extensions...
/content/drive/My Drive/
/content/drive/My Drive//RVG_Merged_Dataset
/content/drive/My Drive//RVG_Merged_Dataset/images
/content/drive/My Drive//RVG_Merged_Dataset/masks_cleaned


In [239]:
len(os.listdir(IMAGE_DIR)), len(os.listdir(MASK_DIR))

(433, 434)

In [240]:
IMG_SIZE = (512, 512)
BATCH_SIZE = 2
AUTOTUNE = tf.data.AUTOTUNE

In [241]:
def load_image(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.cast(img, tf.float32) / 255.0

    img = tf.pad(
        img,
        paddings=[[5, 5], [10, 10], [0, 0]],
        mode="CONSTANT",
        constant_values=0.0
    )

    return img

def load_mask(mask_path):
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.cast(mask, tf.int32)
    mask = tf.pad(
        mask,
        paddings=[[5, 5], [10, 10], [0, 0]],
        mode="CONSTANT",
        constant_values=0
    )
    return mask

In [242]:
def make_binary_mask(mask, class_id):
    return tf.cast(mask == class_id, tf.float32)

In [243]:
def load_pair(img_path, mask_path, class_id):
    img  = load_image(img_path)
    mask = load_mask(mask_path)
    mask = make_binary_mask(mask, class_id)

    return img, mask

In [244]:
def get_matched_pairs():
    img_files = os.listdir(IMAGE_DIR)
    mask_files = os.listdir(MASK_DIR)

    img_map = {f.replace(".png", ""): os.path.join(IMAGE_DIR, f)
               for f in img_files}

    mask_map = {f.replace("_combined_mask.png", ""): os.path.join(MASK_DIR, f)
                for f in mask_files}

    common_keys = sorted(img_map.keys() & mask_map.keys())

    img_paths  = [img_map[k] for k in common_keys]
    mask_paths = [mask_map[k] for k in common_keys]

    return img_paths, mask_paths


In [245]:
def keep_only_correct_shape(img, mask):
    img_shape  = tf.shape(img)
    mask_shape = tf.shape(mask)

    img_ok = tf.logical_and(
        tf.logical_and(img_shape[0] == 800, img_shape[1] == 1120),
        img_shape[2] == 1
    )

    mask_ok = tf.logical_and(
        tf.logical_and(mask_shape[0] == 800, mask_shape[1] == 1120),
        mask_shape[2] == 1
    )

    return tf.logical_and(img_ok, mask_ok)


In [246]:
def create_binary_dataset(class_id):
    img_paths, mask_paths = get_matched_pairs()

    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))

    ds = ds.map(
        lambda x, y: load_pair(x, y, class_id),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return ds

In [247]:
enamel_ds  = create_binary_dataset(class_id=1)
dentine_ds = create_binary_dataset(class_id=2)
pulp_ds    = create_binary_dataset(class_id=3)

enamel_ds = enamel_ds.filter(keep_only_correct_shape)
dentine_ds = dentine_ds.filter(keep_only_correct_shape)
pulp_ds = pulp_ds.filter(keep_only_correct_shape)

In [248]:
enamel_ds = (
    enamel_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
dentine_ds = (
    dentine_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
pulp_ds = (
    pulp_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [249]:
N = sum(1 for _ in enamel_ds)

In [250]:
n_train = int(0.8 * N)
n_val   = int(0.1 * N)
n_test  = N - n_train - n_val
print("train: ", n_train)
print("val: ", n_val)
print("test: ", n_test)

train:  105
val:  13
test:  14


In [251]:
def split_ds(ds, nt, nv, ntt):
  return ds.take(nt), ds.take(nv), ds.take(ntt)

In [252]:
enamel_train, enamel_val, enamel_test = split_ds(enamel_ds, n_train, n_val, n_test)
dentine_train, dentine_val, dentine_test = split_ds(dentine_ds, n_train, n_val, n_test)
pulp_train, pulp_val, pulp_test = split_ds(pulp_ds, n_train, n_val, n_test)

## 4. Model arch

In [253]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x

def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D((2, 2))(c)
    return c, p

def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, (2, 2), strides=(2, 2), padding="same")(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

In [254]:
def build_unet(input_shape=(None, None, 1)):
    inputs = layers.Input(input_shape)

    s1, p1 = encoder_block(inputs,   32)
    s2, p2 = encoder_block(p1,  64)
    s3, p3 = encoder_block(p2, 128)
    s4, p4 = encoder_block(p3, 256)

    b = conv_block(p4, 512)

    d1 = decoder_block(b,  s4, 256)
    d2 = decoder_block(d1, s3, 128)
    d3 = decoder_block(d2, s2,  64)
    d4 = decoder_block(d3, s1,  32)

    outputs = layers.Conv2D(1, (1, 1), activation="sigmoid")(d4)

    return Model(inputs, outputs)

## 5. Crazy loss functions

In [261]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)
    denom = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)

    dice = (2. * intersection + smooth) / (denom + smooth)
    return 1. - dice

def scce_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_true = tf.squeeze(y_true, axis=-1)

    y_pred = tf.concat([1 - y_pred, y_pred], axis=-1)

    return tf.keras.losses.sparse_categorical_crossentropy(
        y_true, y_pred
    )

def focal_tversky_loss(y_true, y_pred,
                       alpha=0.4,
                       beta=0.6,
                       gamma=0.75,
                       smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    TP = tf.reduce_sum(y_true * y_pred)
    FP = tf.reduce_sum((1 - y_true) * y_pred)
    FN = tf.reduce_sum(y_true * (1 - y_pred))

    tversky = (TP + smooth) / (
        TP + alpha * FP + beta * FN + smooth
    )

    return tf.pow((1 - tversky), gamma)

def total_variation_loss(y_pred):
    return tf.reduce_mean(tf.image.total_variation(y_pred))

In [262]:
def combined_loss(y_true, y_pred):

    dice = dice_loss(y_true, y_pred)
    tversky = focal_tversky_loss(y_true, y_pred, alpha=0.3, beta=0.7)
    scce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    # tv = total_variation_loss(y_pred)

    return (
        dice +
        tversky +
        0.5 * scce
    )


## 6. Crazy metrics to track everything

In [263]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)
    denom = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)

    return (2. * intersection + smooth) / (denom + smooth)

def recall_sensitivity(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    TP = tf.reduce_sum(y_true * y_pred)
    FN = tf.reduce_sum(y_true * (1 - y_pred))

    return (TP + smooth) / (TP + FN + smooth)

def precision_metric(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    TP = tf.reduce_sum(y_true * y_pred)
    FP = tf.reduce_sum((1 - y_true) * y_pred)

    return (TP + smooth) / (TP + FP + smooth)

def hd95_metric(y_true, y_pred, threshold=0.5):
    """
    Computes the 95th percentile Hausdorff Distance
    """

    def _hd95(y_true_np, y_pred_np):
        y_true_np = y_true_np.astype(np.bool_)
        y_pred_np = (y_pred_np > threshold)

        if not np.any(y_true_np) and not np.any(y_pred_np):
            return np.array(0.0, dtype=np.float32)

        if not np.any(y_true_np) or not np.any(y_pred_np):
            return np.array(1e6, dtype=np.float32)

        dt_true = distance_transform_edt(~y_true_np)
        dt_pred = distance_transform_edt(~y_pred_np)

        surface_pred = dt_true[y_pred_np]
        surface_true = dt_pred[y_true_np]

        hd95 = np.percentile(
            np.concatenate([surface_pred, surface_true]),
            95
        )

        return np.array(hd95, dtype=np.float32)

    return tf.numpy_function(
        _hd95,
        [tf.squeeze(y_true), tf.squeeze(y_pred)],
        tf.float32
    )

## 7. Callbacks (Not crazy)

In [264]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

## 8. Model building

In [265]:
enamel_model = build_unet()

enamel_model.compile(
    optimizer="adam",
    loss=combined_loss,
    metrics=[
        dice_coefficient,
        recall_sensitivity,
        precision_metric
        #hd95_metric  # Super slow
    ]
)

enamel_model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_11      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_172 (Conv2D) │ (None, None,      │        320 │ input_layer_11[0… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_173 (Conv2D) │ (None, None,      │      9,248 │ conv2d_172[0][0]  │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_40    │ (None, None,      │          0 │ conv2d_173[0][0]  │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_174 (Conv2D) │ (None, None,      │     18,496 │ max_pooling2d_40… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_175 (Conv2D) │ (None, None,      │     36,928 │ conv2d_174[0][0]  │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_41    │ (None, None,      │          0 │ conv2d_175[0][0]  │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_176 (Conv2D) │ (None, None,      │     73,856 │ max_pooling2d_41… │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_177 (Conv2D) │ (None, None,      │    147,584 │ conv2d_176[0][0]  │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_42    │ (None, None,      │          0 │ conv2d_177[0][0]  │
│ (MaxPooling2D)      │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_178 (Conv2D) │ (None, None,      │    295,168 │ max_pooling2d_42… │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_179 (Conv2D) │ (None, None,      │    590,080 │ conv2d_178[0][0]  │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_43    │ (None, None,      │          0 │ conv2d_179[0][0]  │
│ (MaxPooling2D)      │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_180 (Conv2D) │ (None, None,      │  1,180,160 │ max_pooling2d_43… │
│                     │ None, 512)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_181 (Conv2D) │ (None, None,      │  2,359,808 │ conv2d_180[0][0]  │
│                     │ None, 512)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_34 │ (None, None,      │    524,544 │ conv2d_181[0][0]  │
│ (Conv2DTranspose)   │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_34      │ (None, None,      │          0 │ conv2d_transpose

 Total params: 7,759,521 (29.60 MB)

 Trainable params: 7,759,521 (29.60 MB)

 Non-trainable params: 0 (0.00 B)

## 9. Model training

In [ ]:
enamel_history = enamel_model.fit(
    enamel_train,
    validation_data=enamel_val,
    epochs=50,
    # callbacks=callbacks,
    verbose=1
)

Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 68s 529ms/step - dice_coefficient: 0.2743 - loss: 1.6904 - precision_metric: 0.6983 - recall_sensitivity: 0.2525 - val_dice_coefficient: 0.5836 - val_loss: 1.0755 - val_precision_metric: 0.4866 - val_recall_sensitivity: 0.7569
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 53s 505ms/step - dice_coefficient: 0.5265 - loss: 1.1866 - precision_metric: 0.5180 - recall_sensitivity: 0.6831 - val_dice_coefficient: 0.6283 - val_loss: 0.9708 - val_precision_metric: 0.6008 - val_recall_sensitivity: 0.7126
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 53s 507ms/step - dice_coefficient: 0.5676 - loss: 1.0895 - precision_metric: 0.5457 - recall_sensitivity: 0.6908 - val_dice_coefficient: 0.6681 - val_loss: 0.8624 - val_precision_metric: 0.6260 - val_recall_sensitivity: 0.7704
Epoch 4/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 53s 506ms/step - dice_coefficient: 0.6350 - loss: 0.9397 - precision_metric: 0.5931 - recall_sensitivity: 0.7514 - val_dice_coefficient: 0.6593 - val_loss: 0.8

In [ ]:
history = enamel_history.history
epochs = range(1, len(history['loss']) + 1)

plt.figure(figsize=(14, 10))

# 1️⃣ Dice
plt.subplot(2, 2, 1)
plt.plot(epochs, history['dice_coefficient'], label='Train Dice')
plt.plot(epochs, history['val_dice_coefficient'], label='Val Dice')
plt.title('Dice Coefficient')
plt.xlabel('Epoch')
plt.ylabel('Dice')
plt.legend()
plt.grid(True)

# 2️⃣ Loss
plt.subplot(2, 2, 2)
plt.plot(epochs, history['loss'], label='Train Loss')
plt.plot(epochs, history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 3️⃣ Precision
plt.subplot(2, 2, 3)
plt.plot(epochs, history['precision_metric'], label='Train Precision')
plt.plot(epochs, history['val_precision_metric'], label='Val Precision')
plt.title('Precision')
plt.xlabel('Epoch')
plt.ylabel('Precision')
plt.legend()
plt.grid(True)

# 4️⃣ Recall
plt.subplot(2, 2, 4)
plt.plot(epochs, history['recall_sensitivity'], label='Train Recall')
plt.plot(epochs, history['val_recall_sensitivity'], label='Val Recall')
plt.title('Recall (Sensitivity)')
plt.xlabel('Epoch')
plt.ylabel('Recall')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
test_results = enamel_model.evaluate(enamel_test, verbose=1)

test_metrics = dict(zip(enamel_model.metrics_names, test_results))
test_metrics

In [ ]:
images, gts = next(iter(enamel_test))
preds = enamel_model.predict(images)

batch_size = images.shape[0]
threshold = 0.2   # try 0.3 later if recall is low
alpha = 0.5       # transparency for overlays

plt.figure(figsize=(15, 4 * batch_size))

for i in range(batch_size):
    img  = images[i, ..., 0]
    gt   = gts[i, ..., 0]
    pred = preds[i, ..., 0]

    # 1️⃣ Image only
    plt.subplot(batch_size, 3, i * 3 + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"Image {i}")
    plt.axis("off")

    # 2️⃣ Image + GT overlay
    plt.subplot(batch_size, 3, i * 3 + 2)
    plt.imshow(img, cmap="gray")
    plt.imshow(gt, cmap="Reds", alpha=alpha)
    plt.title(f"Image + GT {i}")
    plt.axis("off")

    # 3️⃣ Image + Prediction overlay
    plt.subplot(batch_size, 3, i * 3 + 3)
    plt.imshow(img, cmap="gray")
    plt.imshow(pred > threshold, cmap="Blues", alpha=alpha)
    plt.title(f"Image + Pred {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()
